In [ ]:
import duckdb
con = duckdb.connect("../data/capstone_data.duckdb")


In [2]:
con.execute("DESCRIBE transactions_v2").df()

,column_name,column_type,null,key,default,extra
0,ATM_Address,VARCHAR,YES,None,None,None
1,ATM_City_State,VARCHAR,YES,None,None,None
2,Merchant_Category,VARCHAR,YES,None,None,None
3,Merchant_Name,VARCHAR,YES,None,None,None
4,Transaction_Code,BIGINT,YES,None,None,None
5,Transaction_Code_Description,VARCHAR,YES,None,None,None
6,Transaction_Type,VARCHAR,YES,None,None,None
7,Transaction_Type_Description,VARCHAR,YES,None,None,None
8,Transaction_Number,VARCHAR,YES,None,None,None
9,Amount_Completed,DOUBLE,YES,None,None,None


### Range of data - transactions and customers

In [3]:
query = con.execute("""
    SELECT 
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT CustomerID) AS unique_customers,
        MIN(Transaction_Date) AS earliest_date,
        MAX(Transaction_Date) AS latest_date
    FROM transactions_v2
""").fetchone()

print(f"Total transactions:  {query[0]:,}")
print(f"Unique customers:    {query[1]:,}")
print(f"Date range:          {query[2]} to {query[3]}")

Total transactions:  8,234,091
Unique customers:    128,535
Date range:          2025-09-08 to 2026-01-24


### Amount of transactions

In [4]:
# Transaction amount distribution
query = con.execute("""
    SELECT 
        ROUND(MIN(Amount_Completed), 2) AS min_amt,
        ROUND(MAX(Amount_Completed), 2) AS max_amt,
        ROUND(AVG(Amount_Completed), 2) AS avg_amt,
        ROUND(MEDIAN(Amount_Completed), 2) AS median_amt
    FROM transactions_v2
""").fetchone()

print(f"Min:    ${query[0]:,}")
print(f"Max:    ${query[1]:,}")
print(f"Mean:   ${query[2]:,}")
print(f"Median: ${query[3]:,}")

Min:    $0.0
Max:    $95,000.0
Mean:   $49.5
Median: $21.11


### Extremes : Zero Dollars - Failed/Refund? 

In [5]:
# Checking $0 transactions
zero = con.execute("""
    SELECT COUNT(*) FROM transactions_v2 WHERE Amount_Completed = 0
""").fetchone()[0]

# Checking large transactions >$10k
large = con.execute("""
    SELECT COUNT(*) FROM transactions_v2 WHERE Amount_Completed > 10000
""").fetchone()[0]

total = con.execute("""
    SELECT COUNT(*) FROM transactions_v2
""").fetchone()[0]

print(f"$0 transactions:     {zero:,} ({zero/total*100:.2f}%)")
print(f">$10k transactions:  {large:,} ({large/total*100:.2f}%)")

$0 transactions:     96,716 (1.17%)
>$10k transactions:  137 (0.00%)


### Weekly Classification (DuckDb inbuilt functions)

In [6]:
# Transactions by day of week
dow = con.execute("""
    SELECT 
        DAYNAME(Transaction_Date) AS day_name,
        COUNT(*) AS count
    FROM transactions_v2
    GROUP BY DAYOFWEEK(Transaction_Date), day_name
    ORDER BY DAYOFWEEK(Transaction_Date)
""").df()
print(dow)

print("\n")
# Transactions by hour (using FLOOR as division was getting rounded off to the upper boundary)
hours = con.execute("""
    SELECT 
        FLOOR(Time_Local_hhmmss / 10000) AS hour,
        COUNT(*) AS count
    FROM transactions_v2
    GROUP BY hour
    ORDER BY hour
""").df()
print(hours)

    day_name    count
0     Sunday  1066483
1     Monday  1160346
2    Tuesday  1194986
3  Wednesday  1203250
4   Thursday   983229
5     Friday  1363441
6   Saturday  1262356


    hour   count
0    0.0  133344
1    1.0   94558
2    2.0   88260
3    3.0   95280
4    4.0  117881
5    5.0  164806
6    6.0  225422
7    7.0  306572
8    8.0  373132
9    9.0  449696
10  10.0  523462
11  11.0  607637
12  12.0  648451
13  13.0  608023
14  14.0  592377
15  15.0  592506
16  16.0  592329
17  17.0  529101
18  18.0  414605
19  19.0  322493
20  20.0  256885
21  21.0  205387
22  22.0  157352
23  23.0  134532


### Transactions compared to different merchants

In [7]:
# Top 10 merchant categories
mcc = con.execute("""
    SELECT 
        Merchant_Category,
        COUNT(*) AS count,
        ROUND(AVG(Amount_Completed), 2) AS avg_amount
    FROM transactions_v2
    GROUP BY Merchant_Category
    ORDER BY count DESC
    LIMIT 10
""").df()
print(mcc)

  Merchant_Category    count  avg_amount
0              5411  1354757       53.38
1              5814   972412       13.02
2              5541   780466       19.13
3              5812   490573       41.40
4              5542   414373       36.28
5              5818   293027       14.12
6              5310   254534       55.73
7              4899   236487       32.05
8              5942   218659       47.14
9              5499   150502       27.85


### Transaction Types

In [8]:
# Transaction types breakdown
types = con.execute("""
    SELECT 
        Transaction_Type,
        Transaction_Type_Description,
        COUNT(*) AS count
    FROM transactions_v2
    GROUP BY Transaction_Type, Transaction_Type_Description
    ORDER BY count DESC
""").df()
print(types)

  Transaction_Type Transaction_Type_Description    count
0                X         Pinless Settlements   4831271
1                P         Pinned Settlements    3402820


In [9]:
# Recurring vs one-time transactions
recurring = con.execute("""
    SELECT 
        Recurring_Trxn,
        COUNT(*) AS count
    FROM transactions_v2
    GROUP BY Recurring_Trxn
    ORDER BY count DESC
""").df()
print(recurring)

  Recurring_Trxn    count
0                 7285574
1              Y   948517


In [10]:
# Transactions per customer
tpc = con.execute("""
    SELECT 
        MIN(txn_count) AS min_txns,
        MAX(txn_count) AS max_txns,
        ROUND(AVG(txn_count), 1) AS avg_txns,
        ROUND(MEDIAN(txn_count), 1) AS median_txns
    FROM (
        SELECT CustomerID, COUNT(*) AS txn_count
        FROM transactions_v2
        GROUP BY CustomerID
    )
""").fetchone()

print(f"Min:    {tpc[0]:,}")
print(f"Max:    {tpc[1]:,}")
print(f"Mean:   {tpc[2]:,}")
print(f"Median: {tpc[3]:,}")

Min:    1
Max:    1,425
Mean:   64.1
Median: 46.0


### Customers having highest number of transactions

In [11]:
tpc = con.execute("""
        SELECT CustomerID, COUNT(*) AS txn_count
        FROM transactions_v2
        GROUP BY CustomerID
        ORDER BY txn_count DESC
""").df()

print(tpc)

       CustomerID  txn_count
0       BSB293802       1425
1       BSB334522       1380
2       BSB036028       1274
3       BSB059995       1122
4       BSB501141       1097
...           ...        ...
128530  BSB457713          1
128531  BSB420693          1
128532  BSB447459          1
128533  BSB132989          1
128534  BSB085767          1

[128535 rows x 2 columns]


### BSB293802 - 1425 transactions in 2 months!

In [12]:
tpc = con.execute("""
    SELECT 
        MIN(Transaction_Date) AS first_txn,
        MAX(Transaction_Date) AS last_txn,
        COUNT(*) AS total_txns
    FROM transactions_v2
    WHERE CustomerID = 'BSB293802'
""").df()

print(tpc)

   first_txn   last_txn  total_txns
0 2025-11-20 2026-01-22        1425


### Missing values - ATM_City_State - Online transactions?

In [13]:
# Missing values check for transactions
cols = con.execute("DESCRIBE transactions_v2").df()['column_name'].tolist()

null_checks = ', '.join([f"COUNT(*) - COUNT({c}) AS {c}" for c in cols])
missing = con.execute(f"SELECT {null_checks} FROM transactions_v2").fetchone()

has_missing = False
for col, nulls in zip(cols, missing):
    if nulls > 0:
        print(f"  {col}: {nulls:,} missing")
        has_missing = True

if not has_missing:
    print("  No missing values found")

  ATM_City_State: 23 missing


### Some more EDA -

In [14]:
txn = con.execute("""
                        SELECT 
                        customerid,
                        MIN(transaction_date) AS first_txn,
                        MAX(transaction_date) AS last_txn,
                        DATEDIFF('month', MIN(transaction_date), MAX(transaction_date)) AS months_active,
                        COUNT(*) AS txn_count
                        FROM transactions_v2
                        GROUP BY customerid
                        ORDER BY txn_count DESC
                    """).df()

print(txn)

       CustomerID  first_txn   last_txn  months_active  txn_count
0       BSB293802 2025-11-20 2026-01-22              2       1425
1       BSB334522 2025-11-19 2026-01-22              2       1380
2       BSB036028 2025-11-20 2026-01-22              2       1274
3       BSB059995 2025-11-21 2026-01-22              2       1122
4       BSB501141 2025-11-19 2026-01-22              2       1097
...           ...        ...        ...            ...        ...
128530  BSB293560 2026-01-09 2026-01-09              0          1
128531  BSB415054 2025-11-28 2025-11-28              0          1
128532  BSB305261 2025-12-07 2025-12-07              0          1
128533  BSB475461 2026-01-04 2026-01-04              0          1
128534  BSB412896 2026-01-19 2026-01-19              0          1

[128535 rows x 5 columns]


In [15]:
distribution = con.execute(""" SELECT 
                                CASE 
                                    WHEN txn_count < 10 THEN '<10'
                                    WHEN txn_count < 50 THEN '10-49'
                                    WHEN txn_count < 100 THEN '50-99'
                                    WHEN txn_count < 500 THEN '100-499'
                                    ELSE '500+'
                                END AS txn_bucket,
                                COUNT(*) AS customers
                                FROM (
                                SELECT customerid, COUNT(*) AS txn_count
                                FROM transactions_v2
                                GROUP BY 1
                                )
                                GROUP BY 1
                                ORDER BY 1;
                           """).df()

print(distribution)

  txn_bucket  customers
0      10-49      39764
1    100-499      30949
2      50-99      30487
3       500+         53
4        <10      27282


In [16]:
txn_m = con.execute("""
                        SELECT 
                        DATE_TRUNC('month', transaction_date) AS month,
                        COUNT(*) AS txns,
                        COUNT(DISTINCT customerid) AS unique_customers
                        FROM transactions_v2
                        GROUP BY 1
                        ORDER BY 1;
                    """).df()

print(txn_m)

       month     txns  unique_customers
0 2025-09-01        2                 2
1 2025-10-01        8                 8
2 2025-11-01  1314573            106869
3 2025-12-01  4310823            123174
4 2026-01-01  2608685            117449


In [17]:
query = con.execute("""SELECT * FROM transactions_v2 WHERE ATM_City_State is NULL""").df()

print(query.T)

                                                          0   \
ATM_Address                   Room B3, 19/F, Tung,SHEUNG WAN   
ATM_City_State                                          None   
Merchant_Category                                       5691   
Merchant_Name                                     SP ATTIFIT   
Transaction_Code                                          40   
Transaction_Code_Description                  D/C SETTLEMENT   
Transaction_Type                                           X   
Transaction_Type_Description            Pinless Settlements    
Transaction_Number                                        66   
Amount_Completed                                       26.08   
Transaction_Date                         2025-11-28 00:00:00   
Time_Local_hhmmss                                      71453   
Recurring_Trxn                                                 
CustomerID                                         BSB005675   

                                       

In [5]:
query = con.execute("""SELECT *
FROM transactions_v2 
WHERE ATM_City_State IS NULL 
   OR ATM_City_State = '' 
   OR ATM_City_State = 'NA'
""").df()

print(query.T)

                                                0                     1   \
ATM_Address                    90 WOODLANDS INDUST            Hurst Lane   
ATM_City_State                                  NA                    NA   
Merchant_Category                             5817                  8911   
Merchant_Name                             Netshort                    NA   
Transaction_Code                                40                    40   
Transaction_Code_Description        D/C SETTLEMENT        D/C SETTLEMENT   
Transaction_Type                                 X                     X   
Transaction_Type_Description  Pinless Settlements   Pinless Settlements    
Transaction_Number                              95                    20   
Amount_Completed                              20.0                141.54   
Transaction_Date               2025-11-27 00:00:00   2025-11-24 00:00:00   
Time_Local_hhmmss                           231044                181144   
Recurring_Tr

In [6]:
query_output = '/home/omkar/ds5500/data/processed/queries/null_atmcitystate.csv'
query.to_csv(query_output, index=False)
print(f"Exported {len(query)} rows to {query_output}")

Exported 90 rows to /home/omkar/ds5500/data/processed/queries/null_atmcitystate.csv


In [8]:
query = con.execute("""SELECT * FROM transactions_v2 WHERE Amount_Completed = 0""").df()

print(query.T)

                                             0                     1      \
ATM_Address                     CIRCLE K 07113 719      10305 PACIFIC ST   
ATM_City_State                       MILLINOCKETME               OMAHANE   
Merchant_Category                             5541                  5411   
Merchant_Name                     CIRCLE K 07113 7      TRADER JOE S #71   
Transaction_Code                                20                    20   
Transaction_Code_Description  POS Debit        Pri  POS Debit        Pri   
Transaction_Type                                 P                     P   
Transaction_Type_Description  Pinned Settlements    Pinned Settlements     
Transaction_Number                              30                    62   
Amount_Completed                               0.0                   0.0   
Transaction_Date               2025-11-23 00:00:00   2025-11-24 00:00:00   
Time_Local_hhmmss                           213507                165518   
Recurring_Tr

In [9]:
query_output = '/home/omkar/ds5500/data/processed/queries/zero_transactions.csv'
query.to_csv(query_output, index=False)
print(f"Exported {len(query)} rows to {query_output}")

Exported 96716 rows to /home/omkar/ds5500/data/processed/queries/zero_transactions.csv


In [6]:
transaction_type = con.execute("""SELECT COUNT(*) AS Number, Transaction_Type FROM transactions_v2 GROUP BY Transaction_Type  """).df()
print(transaction_type)

    Number Transaction_Type
0  3402820                P
1  4831271                X


In [8]:
transaction_code = con.execute("""SELECT COUNT(*) AS Number, Transaction_Code FROM transactions_v2 GROUP BY Transaction_Code  """).df()
print(transaction_code)

    Number  Transaction_Code
0    57715                 0
1  3345105                20
2  4611094                40
3   220177              3120


In [9]:
transaction_code_desc = con.execute("""SELECT COUNT(*) AS Number, Transaction_Code_Description FROM transactions_v2 GROUP BY Transaction_Code_Description  """).df()
print(transaction_code_desc)

    Number Transaction_Code_Description
0   220177         Pinless Bill Payment
1  3402820         POS Debit        Pri
2  4611094               D/C SETTLEMENT
